# Funding Eligibility Finder

## Notebook 2 — Funding Extractor

### Objective

This notebook takes the selected funding candidates generated by Notebook 1 and retrieves information from each opportunity page.

The goal is to transform search results into structured funding information by extracting:

- page retrieval status;
- readable page text length;
- possible funding amounts;
- amount-related snippets;
- deadline-related snippets;
- parsed deadlines;
- eligibility-related snippets;
- application-related links.

This notebook represents the second stage of the Funding Eligibility Finder pipeline.

---

### Input

`results/funding_candidates_for_extraction.csv`

Generated by Notebook 1.

---

### Output

- `results/funding_extracted_opportunities.csv`
- `results/funding_extracted_opportunities.xlsx`
- `results/funding_extracted_useful_only.csv`
- `results/funding_extracted_useful_only.xlsx`

---

### Workflow

1. Load selected candidates
2. Retrieve web pages
3. Extract readable text
4. Extract possible funding amounts
5. Extract amount-related snippets
6. Extract deadline-related snippets
7. Parse possible deadlines
8. Extract eligibility-related snippets
9. Extract application-related links
10. Export structured results

In [ ]:
!pip install -q pandas requests beautifulsoup4 trafilatura dateparser openpyxl tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.6/134.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.5/300.5 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 20.0 MB/s eta 0:00:00


In [1]:
# ============================================================
# LOAD INPUT FILE FROM NOTEBOOK 1
# ============================================================

from google.colab import files
import pandas as pd

print("Upload the file generated by Notebook 1:")
print("funding_candidates_for_extraction.csv\n")

uploaded = files.upload()

if len(uploaded) == 0:
    raise FileNotFoundError("No file was uploaded.")

# Automatically detect the uploaded file
INPUT_FILE = next(iter(uploaded.keys()))

print(f"\nInput file detected: {INPUT_FILE}")

# Load dataset
candidates_check = pd.read_csv(INPUT_FILE)

print(f"Loaded {len(candidates_check)} candidate opportunities.")
print(f"Dataset shape: {candidates_check.shape}")

display(candidates_check.head())

Upload the file generated by Notebook 1:
funding_candidates_for_extraction.csv



Saving funding_candidates_for_extraction-4.csv to funding_candidates_for_extraction-4.csv

Input file detected: funding_candidates_for_extraction-4.csv
Loaded 60 candidate opportunities.
Dataset shape: (60, 12)


,query,title,url,snippet,status,relevance_score,is_aggregator,funding_matches,profile_matches,negative_flags,course_page_flags,selected_for_extraction
0,female finance students scholarship,Top 2026 Scholarships for Women: College Fundi...,https://www.fastweb.com/college-scholarships/a...,Explore the best 2026 scholarships for women a...,high potential,27,False,"scholarship, scholarships, grant, grants, fund...","women, female, stem",NaN,NaN,True
1,women in financial mathematics scholarship,Funding & Scholarships for Women in Mathematic...,https://www.mathunion.org/cwm/resources/fundin...,Explore funding opportunities and scholarships...,high potential,25,False,"scholarship, scholarships, grant, grants, fund...",women,NaN,NaN,True
2,funding award for Italian students in data sci...,Spärck AI Scholarship | UCL Scholarships and f...,https://www.ucl.ac.uk/scholarships/sparck-ai-s...,The Spärck AI Scholarship may not be held alon...,high potential,23,False,"scholarship, scholarships, bursary, bursaries,...","master, uk",NaN,NaN,True
3,scholarships for female mathematicians,AMS :: Fellowships & Scholarships - American M...,https://www.ams.org/grants-awards/ams-fellowsh...,The Joan and Joseph Birman Fellowship for the ...,high potential,22,False,"scholarship, scholarships, grant, grants, awar...",women,NaN,NaN,True
4,funding for Italian students pursuing a master...,Scholarships for Master's Studies in the Unite...,https://www.educations.com/articles-and-advice...,The UK Research and Innovation (UKRI) offers v...,high potential,20,False,"scholarship, scholarships, grant, grants, funding","international students, postgraduate, master, ...",NaN,NaN,True


In [2]:
# ============================================================
# FUNDING ELIGIBILITY FINDER
# Notebook 2: Funding Extractor
# ============================================================

!pip install -q requests beautifulsoup4 trafilatura dateparser openpyxl tqdm

import logging
import re
import time
from pathlib import Path
from urllib.parse import urljoin

import dateparser
import pandas as pd
import requests
import trafilatura

from bs4 import BeautifulSoup
from tqdm import tqdm


# ============================================================
# LOGGING
# ============================================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)

log = logging.getLogger("funding_extractor")


# ============================================================
# CONFIGURATION
# ============================================================

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

OUTPUT_CSV = RESULTS_DIR / "funding_extracted_opportunities.csv"
OUTPUT_EXCEL = RESULTS_DIR / "funding_extracted_opportunities.xlsx"

USEFUL_OUTPUT_CSV = RESULTS_DIR / "funding_extracted_useful_only.csv"
USEFUL_OUTPUT_EXCEL = RESULTS_DIR / "funding_extracted_useful_only.xlsx"

REQUEST_TIMEOUT = 10
PAUSE_SECONDS = 0.7
MAX_PAGES = None
MIN_USEFUL_TEXT_LENGTH = 300

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (compatible; FundingEligibilityFinder/1.0; "
        "+https://github.com/elisabattista)"
    )
}


# ============================================================
# KEYWORDS AND PATTERNS
# ============================================================

DEADLINE_KEYWORDS = [
    "deadline", "closing date", "application deadline",
    "applications close", "applications closing", "apply by",
    "submit by", "submission deadline", "closing deadline",
    "last date", "due date", "accepting applications until",
    "applications are due", "applications must be submitted by",
    "closing on"
]

DATE_PATTERNS = [
    r"\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b",
    r"\b\d{1,2}\s+(?:January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{4}\b",
    r"\b(?:January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{1,2},?\s+\d{4}\b",
    r"\b\d{1,2}\s+(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Sept|Oct|Nov|Dec)\s+\d{4}\b",
    r"\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Sept|Oct|Nov|Dec)\s+\d{1,2},?\s+\d{4}\b"
]

AMOUNT_PATTERNS = [
    r"£\s?\d{1,3}(?:,\d{3})*(?:\.\d+)?",
    r"€\s?\d{1,3}(?:,\d{3})*(?:\.\d+)?",
    r"\$\s?\d{1,3}(?:,\d{3})*(?:\.\d+)?",
    r"\d{1,3}(?:,\d{3})*\s?(?:GBP|EUR|USD)",
    r"full tuition fee",
    r"tuition fees",
    r"living expenses",
    r"maintenance support",
    r"stipend"
]

AMOUNT_CONTEXT_KEYWORDS = [
    "amount", "value", "worth", "award", "scholarship",
    "grant", "bursary", "funding", "tuition",
    "stipend", "living expenses", "maintenance"
]

ELIGIBILITY_KEYWORDS = [
    "eligibility", "eligible", "criteria", "requirements",
    "applicants must", "applicants should", "open to",
    "available to", "must be", "international students",
    "women", "female", "postgraduate", "master", "msc",
    "nationality", "citizenship", "resident", "country",
    "domiciled", "enrolled", "admitted"
]

APPLICATION_LINK_KEYWORDS = [
    "apply", "application", "how to apply",
    "submit", "register", "application form"
]

AGGREGATOR_SIGNALS = [
    "list of scholarships", "top scholarships",
    "scholarships by country", "scholarships for international students",
    "verified links", "scholarship directory", "browse scholarships",
    "search scholarships", "scholarship database",
    "opportunities database", "find scholarships"
]

COURSE_PAGE_SIGNALS = [
    "course overview", "programme structure", "program structure",
    "entry requirements", "modules", "curriculum",
    "course details", "study this course", "apply for this course"
]

SPECIFIC_OPPORTUNITY_SIGNALS = [
    "scholarship overview", "award value", "application deadline",
    "how to apply", "eligibility criteria", "applications close",
    "applications are open", "apply now", "scholarship amount",
    "who can apply"
]


# ============================================================
# UTILITY FUNCTIONS
# ============================================================

def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def clean_lower(text):
    return clean_text(text).lower()


def unique_preserve_order(items):
    return list(dict.fromkeys(items))


def split_sentences(text):
    if not text:
        return []

    sentences = re.split(r"(?<=[.!?])\s+", text)

    return [
        sentence.strip()
        for sentence in sentences
        if len(sentence.strip()) > 20
    ]


# ============================================================
# LOAD DATA
# ============================================================

def load_candidates(path):
    df = pd.read_csv(path)

    required_columns = ["title", "url"]
    missing_columns = [
        column for column in required_columns
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")

    df = (
        df
        .dropna(subset=["url"])
        .drop_duplicates(subset=["url"])
        .reset_index(drop=True)
    )

    if MAX_PAGES is not None:
        df = df.head(MAX_PAGES)

    log.info("Loaded candidates: %d", len(df))

    return df


# ============================================================
# PAGE RETRIEVAL
# ============================================================

def fetch_page(url):
    try:
        response = requests.get(
            url,
            headers=HEADERS,
            timeout=REQUEST_TIMEOUT,
            allow_redirects=True
        )

        content_type = response.headers.get("Content-Type", "")
        response.raise_for_status()

        if "text/html" not in content_type.lower():
            return {
                "success": False,
                "html": "",
                "status_code": response.status_code,
                "content_type": content_type,
                "final_url": response.url,
                "error": "Non-HTML content"
            }

        return {
            "success": True,
            "html": response.text,
            "status_code": response.status_code,
            "content_type": content_type,
            "final_url": response.url,
            "error": ""
        }

    except Exception as error:
        return {
            "success": False,
            "html": "",
            "status_code": None,
            "content_type": "",
            "final_url": "",
            "error": str(error)
        }


# ============================================================
# TEXT EXTRACTION
# ============================================================

def extract_text_from_html(html):
    if not html:
        return ""

    extracted = trafilatura.extract(html)

    if extracted:
        return clean_text(extracted)

    soup = BeautifulSoup(html, "html.parser")

    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    text = soup.get_text(separator=" ")

    return clean_text(text)


# ============================================================
# AMOUNT EXTRACTION
# ============================================================

def extract_amounts(text):
    matches = []

    for pattern in AMOUNT_PATTERNS:
        found = re.findall(pattern, text, flags=re.IGNORECASE)
        matches.extend(found)

    matches = [
        clean_text(match)
        for match in matches
        if clean_text(match)
    ]

    return ", ".join(unique_preserve_order(matches)[:12])


def extract_amount_snippets(text):
    sentences = split_sentences(text)
    snippets = []

    for sentence in sentences:
        sentence_lower = sentence.lower()

        has_amount = any(
            re.search(pattern, sentence, flags=re.IGNORECASE)
            for pattern in AMOUNT_PATTERNS
        )

        has_context = any(
            keyword in sentence_lower
            for keyword in AMOUNT_CONTEXT_KEYWORDS
        )

        if has_amount and has_context:
            snippets.append(sentence)

    return " | ".join(unique_preserve_order(snippets)[:8])


# ============================================================
# DEADLINE EXTRACTION
# ============================================================

def extract_deadline_snippets(text):
    sentences = split_sentences(text)
    snippets = []

    for sentence in sentences:
        sentence_lower = sentence.lower()

        if any(keyword in sentence_lower for keyword in DEADLINE_KEYWORDS):
            snippets.append(sentence)

    return unique_preserve_order(snippets)[:10]


def parse_dates_from_snippets(snippets):
    parsed_dates = []

    for snippet in snippets:
        for pattern in DATE_PATTERNS:
            matches = re.finditer(pattern, snippet, flags=re.IGNORECASE)

            for match in matches:
                date_text = match.group(0)

                parsed = dateparser.parse(
                    date_text,
                    settings={
                        "DATE_ORDER": "DMY",
                        "PREFER_DAY_OF_MONTH": "last"
                    }
                )

                if parsed:
                    parsed_dates.append(parsed.date().isoformat())

    return unique_preserve_order(parsed_dates)[:5]


def extract_deadlines(text):
    snippets = extract_deadline_snippets(text)
    parsed_dates = parse_dates_from_snippets(snippets)

    return {
        "deadline_snippets": " | ".join(snippets),
        "parsed_deadlines": ", ".join(parsed_dates)
    }


# ============================================================
# ELIGIBILITY EXTRACTION
# ============================================================

def extract_eligibility_snippets(text):
    sentences = split_sentences(text)
    snippets = []

    for sentence in sentences:
        sentence_lower = sentence.lower()

        if any(keyword in sentence_lower for keyword in ELIGIBILITY_KEYWORDS):
            snippets.append(sentence)

    return " | ".join(unique_preserve_order(snippets)[:10])


# ============================================================
# APPLICATION LINK EXTRACTION
# ============================================================

def extract_application_links(html, base_url):
    if not html:
        return ""

    soup = BeautifulSoup(html, "html.parser")
    links = []

    for a_tag in soup.find_all("a", href=True):
        link_text = clean_text(a_tag.get_text(" ")).lower()
        href = a_tag["href"]

        combined = f"{link_text} {href}".lower()

        if any(keyword in combined for keyword in APPLICATION_LINK_KEYWORDS):
            absolute_url = urljoin(base_url, href)
            links.append(absolute_url)

    return " | ".join(unique_preserve_order(links)[:8])


# ============================================================
# PAGE TYPE SIGNALS
# ============================================================

def detect_page_type_signal(row, extracted_text):
    title = clean_lower(row.get("title", ""))
    url = clean_lower(row.get("url", ""))
    text = clean_lower(extracted_text)

    combined = f"{title} {url} {text}"

    specific_score = sum(
        signal in combined
        for signal in SPECIFIC_OPPORTUNITY_SIGNALS
    )

    aggregator_score = sum(
        signal in combined
        for signal in AGGREGATOR_SIGNALS
    )

    course_score = sum(
        signal in combined
        for signal in COURSE_PAGE_SIGNALS
    )

    if len(extracted_text) < MIN_USEFUL_TEXT_LENGTH:
        return "low text content"

    if specific_score >= 2 and specific_score >= aggregator_score and specific_score >= course_score:
        return "specific opportunity page"

    if aggregator_score >= 2:
        return "aggregator or list page"

    if course_score >= 2:
        return "course page"

    if specific_score >= 1:
        return "possibly specific opportunity page"

    return "unclear"


# ============================================================
# MAIN EXTRACTION FUNCTION
# ============================================================

def process_candidate(row):
    url = row["url"]
    page = fetch_page(url)

    if not page["success"]:
        return pd.Series({
            "page_retrieved": False,
            "status_code": page["status_code"],
            "content_type": page["content_type"],
            "final_url": page["final_url"],
            "fetch_error": page["error"],
            "extracted_text_length": 0,
            "possible_amounts": "",
            "amount_snippets": "",
            "deadline_snippets": "",
            "parsed_deadlines": "",
            "eligibility_snippets": "",
            "application_links": "",
            "page_type_signal": "not retrieved"
        })

    html = page["html"]
    text = extract_text_from_html(html)
    deadline_info = extract_deadlines(text)

    return pd.Series({
        "page_retrieved": True,
        "status_code": page["status_code"],
        "content_type": page["content_type"],
        "final_url": page["final_url"],
        "fetch_error": "",
        "extracted_text_length": len(text),
        "possible_amounts": extract_amounts(text),
        "amount_snippets": extract_amount_snippets(text),
        "deadline_snippets": deadline_info["deadline_snippets"],
        "parsed_deadlines": deadline_info["parsed_deadlines"],
        "eligibility_snippets": extract_eligibility_snippets(text),
        "application_links": extract_application_links(html, page["final_url"] or url),
        "page_type_signal": detect_page_type_signal(row, text)
    })


# ============================================================
# PIPELINE
# ============================================================

def run_extraction_pipeline():
    candidates = load_candidates(INPUT_FILE)

    extracted_rows = []

    for _, row in tqdm(
        candidates.iterrows(),
        total=len(candidates),
        desc="Extracting pages"
    ):
        extracted = process_candidate(row)
        combined = pd.concat([row, extracted])
        extracted_rows.append(combined)

        time.sleep(PAUSE_SECONDS)

    results = pd.DataFrame(extracted_rows)

    results.to_csv(OUTPUT_CSV, index=False)
    results.to_excel(OUTPUT_EXCEL, index=False)

    useful_results = results[
        (
            (results["possible_amounts"] != "") |
            (results["amount_snippets"] != "") |
            (results["deadline_snippets"] != "") |
            (results["eligibility_snippets"] != "") |
            (results["application_links"] != "")
        )
    ].copy()

    useful_results.to_csv(USEFUL_OUTPUT_CSV, index=False)
    useful_results.to_excel(USEFUL_OUTPUT_EXCEL, index=False)

    log.info("Extraction completed.")
    log.info("Saved extracted opportunities: %s", OUTPUT_CSV)
    log.info("Saved useful extracted opportunities: %s", USEFUL_OUTPUT_CSV)
    log.info("Pages retrieved: %d / %d", results["page_retrieved"].sum(), len(results))
    log.info("Useful rows: %d / %d", len(useful_results), len(results))

    return results, useful_results


# ============================================================
# RUN NOTEBOOK 2
# ============================================================

extracted_results, useful_extracted_results = run_extraction_pipeline()

display(
    extracted_results[
        [
            "title",
            "url",
            "final_url",
            "page_retrieved",
            "extracted_text_length",
            "possible_amounts",
            "amount_snippets",
            "deadline_snippets",
            "parsed_deadlines",
            "eligibility_snippets",
            "application_links",
            "page_type_signal"
        ]
    ].head(20)
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.6/134.6 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.5/300.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 16.1 MB/s eta 0:00:00


Extracting pages: 100%|██████████| 60/60 [02:03<00:00,  2.05s/it]


,title,url,final_url,page_retrieved,extracted_text_length,possible_amounts,amount_snippets,deadline_snippets,parsed_deadlines,eligibility_snippets,application_links,page_type_signal
0,Top 2026 Scholarships for Women: College Fundi...,https://www.fastweb.com/college-scholarships/a...,https://www.fastweb.com/college-scholarships/a...,True,19742,"$5,000, $10,000, $15,000, $3,000, $2,000, $25,...",Available to: High School Seniors Award Amount...,Deadline: 3/15/26 Award Amount: Varies The Kri...,,"In honor of Women's History Month, we are high...",https://www.fastweb.com/student-news/articles/...,unclear
1,Funding & Scholarships for Women in Mathematic...,https://www.mathunion.org/cwm/resources/fundin...,https://www.mathunion.org/cwm/resources/fundin...,True,5796,"$60,000",For example: The L’Oréal USA For Women In Scie...,,,Here is some information about funding that re...,https://cws.auburn.edu/shared/files?id=217&fil...,unclear
2,Spärck AI Scholarship | UCL Scholarships and f...,https://www.ucl.ac.uk/scholarships/sparck-ai-s...,https://www.ucl.ac.uk/scholarships/sparck-ai-s...,True,5424,"£22,780, full tuition fee, tuition fees, stipend",- The value of the scholarship is full tuition...,| Programme | Application submission email add...,,Eligibility Candidates must fulfil the followi...,https://view.officeapps.live.com/op/view.aspx?...,possibly specific opportunity page
3,AMS :: Fellowships & Scholarships - American M...,https://www.ams.org/grants-awards/ams-fellowsh...,https://www.ams.org/grants-awards/ams-fellowsh...,True,2855,,,,,- Joan and Joseph Birman Fellowship for the Ad...,https://ebus.ams.org/ebus/Register.aspx?_gl=1*...,unclear
4,Scholarships for Master's Studies in the Unite...,https://www.educations.com/articles-and-advice...,https://www.educations.com/articles-and-advice...,True,4888,"£18,180, £1,515, £5,000, £15,609, £10,000, £1,...",Scholarships are a great way to make your stud...,,,Scholarships for Master's Studies in the Unite...,https://www.educations.com/articles-and-advice...,unclear
5,Women in STEM Scholarships for Studying Overse...,https://www.icanstudent.com/women-in-stem-scho...,https://www.icanstudent.com/women-in-stem-scho...,True,6130,"£40,000, £85,000, $90,000, $140,000, $70,000, ...",Why Women in STEM Scholarships Matter in 2026 ...,How to Apply for Women in STEM Scholarships To...,,"The demand for women in Science, Technology, E...",https://www.icanstudent.com/oci-foundation-cyf...,possibly specific opportunity page
6,University College London Mathematics Scholars...,https://mucuruzi.com/university-college-london...,https://mucuruzi.com/university-college-london...,True,693,,,University College London Mathematics Scholars...,,"Check well before applying, if you doubt about...",https://www.ucl.ac.uk/mathematical-physical-sc...,possibly specific opportunity page
7,UCL Master’s Funding Opportunities 2026/27: Bu...,https://opportunitiesforyouth.org/2026/02/17/u...,https://opportunitiesforyouth.org/2026/02/17/u...,True,3946,"£6,000, £10,000, £42,875, £15,000, tuition fee...","Key Details - Value: £6,000 (one year) - Numbe...","Key Details - Value: £6,000 (one year) - Numbe...","2026-06-25, 2026-05-07",UCL Master’s Funding Opportunities 2026/27: Bu...,https://opportunitiesforyouth.org/2026/06/25/u...,specific opportunity page
8,Master's degree funding | Postgraduate study |...,https://www.lboro.ac.uk/study/postgraduate/fee...,https://www.lboro.ac.uk/study/postgraduate/fee...,True,9583,"£10,000, £5,000, £1,500, £3,000, £5,500, £2,00...",Campus: Both campuses Eligibility: UK and inte...,,,Master's degree funding There are lots of opti...,https://www.lboro.ac.uk/study/postgraduate/app...,unclear
9,UCL Master's Bursary | UCL Scholarships and fu...,https://www.ucl.ac.uk/scholarships/ucl-masters...,https://www.ucl.ac.uk/scholarships/ucl-masters...,True,2644,"£42,875, £10,000, tuition fees",Eligibility Candidates must fulfil all of the ...,The deadline for applications to this scholars...,2026-06-25,The UCL Mas

In [3]:
print("Retrieved pages:")
print(extracted_results["page_retrieved"].value_counts())

print("\nAverage extracted text length:")
print(extracted_results["extracted_text_length"].mean())

print("\nRows with possible amounts:")
print((extracted_results["possible_amounts"] != "").sum())

print("\nRows with amount snippets:")
print((extracted_results["amount_snippets"] != "").sum())

print("\nRows with deadline snippets:")
print((extracted_results["deadline_snippets"] != "").sum())

print("\nRows with parsed deadlines:")
print((extracted_results["parsed_deadlines"] != "").sum())

print("\nRows with eligibility snippets:")
print((extracted_results["eligibility_snippets"] != "").sum())

print("\nRows with application links:")
print((extracted_results["application_links"] != "").sum())

print("\nPage type signals:")
print(extracted_results["page_type_signal"].value_counts())

Retrieved pages:
page_retrieved
True     47
False    13
Name: count, dtype: int64

Average extracted text length:
5234.566666666667

Rows with possible amounts:
36

Rows with amount snippets:
36

Rows with deadline snippets:
30

Rows with parsed deadlines:
13

Rows with eligibility snippets:
47

Rows with application links:
37

Page type signals:
page_type_signal
unclear                               18
possibly specific opportunity page    16
not retrieved                         13
specific opportunity page              9
aggregator or list page                2
low text content                       1
course page                            1
Name: count, dtype: int64


In [5]:
from google.colab import files

files.download(str(OUTPUT_CSV))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>